In [1]:
## READS MNQ-TICK from HGT-Export SC chartbook

In [2]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [3]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/mnq-tick/prod1'

In [5]:
#symbol = 'mnq'
#SEC = 2

########################################
# 07-08
# One file 
# Add logging
# Replace Rex vel to last  
# Run 4 stream to compare to iter3
# run 4 steam with rsx
# Rsx vel on ha last
# Rsx vel on raw hlc
# Rsx vel on raw last
# 
# Run best 8 seed
# 
# Replace d1 d2
# 
# Run 8 seed
########################################

#inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ-TICK-OSCILLATOR-6SEC.csv'
#outFile= f'data/mnq-tick-oscillator-6sec.pqt'
inFile = f'/mnt/d/SierraChart/data/MNQ-6T-NQ-3S-VOL2-V2.txt'
outFile= f'data/mnq-6t-nq-3s-vol2-v2.pqt'

print(inFile, outFile)

/mnt/d/SierraChart/data/MNQ-6T-NQ-3S-VOL2-V2.txt data/mnq-6t-nq-3s-vol2-v2.pqt


In [6]:
df = pd.read_csv(inFile, skipinitialspace=True)
df.columns = df.columns.str.replace(r"\s+", "", regex=True)

#df.to_parquet('txt.pqt', index=False)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11117814 entries, 0 to 11117813
Data columns (total 34 columns):
 #   Column       Dtype  
---  ------       -----  
 0   Date         str    
 1   Time         str    
 2   Open         float64
 3   High         float64
 4   Low          float64
 5   Last         float64
 6   Volume       int64  
 7   #ofTrades    int64  
 8   OHLCAvg      float64
 9   HLCAvg       float64
 10  HLAvg        float64
 11  BidVolume    int64  
 12  AskVolume    int64  
 13  Open.1       float64
 14  High.1       float64
 15  Low.1        float64
 16  Last.1       float64
 17  JMA          float64
 18  VEL          float64
 19  ZeroLevel    float64
 20  RSX          float64
 21  TopLevel     float64
 22  BottomLevel  float64
 23  ZeroLevel.1  float64
 24  Momentum     float64
 25  Line         float64
 26  Momentum.1   float64
 27  Line.1       float64
 28  Momentum.2   float64
 29  Momentum.3   float64
 30  Line.2       float64
 31  JMA.1        float64
 32  CFB    

In [7]:
# expand stupid SC date like 2026-7-8 to 2026-07-08
parts = df['Date'].astype(str).str.split('-', expand=True)

year  = parts[0]
month = parts[1].str.zfill(2)
day   = parts[2].str.zfill(2)

df['Date_norm'] = year + '-' + month + '-' + day

# join date and time and convert to wall clock datetime64
df['timestamp'] = pd.to_datetime(
    df['Date_norm'] + ' ' + df['Time'].astype(str),
    utc=False,            # keep as wall clock, no timezone
    errors='raise'        # or 'coerce' if you want bad rows as NaT
)

# remove temp columns
df = df.drop(columns=['Date', 'Date_norm', 'Time'])

# make timestamp first column
col = df.pop('timestamp')
df.insert(0, 'timestamp', col)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11117814 entries, 0 to 11117813
Data columns (total 33 columns):
 #   Column       Dtype         
---  ------       -----         
 0   timestamp    datetime64[us]
 1   Open         float64       
 2   High         float64       
 3   Low          float64       
 4   Last         float64       
 5   Volume       int64         
 6   #ofTrades    int64         
 7   OHLCAvg      float64       
 8   HLCAvg       float64       
 9   HLAvg        float64       
 10  BidVolume    int64         
 11  AskVolume    int64         
 12  Open.1       float64       
 13  High.1       float64       
 14  Low.1        float64       
 15  Last.1       float64       
 16  JMA          float64       
 17  VEL          float64       
 18  ZeroLevel    float64       
 19  RSX          float64       
 20  TopLevel     float64       
 21  BottomLevel  float64       
 22  ZeroLevel.1  float64       
 23  Momentum     float64       
 24  Line         float64       
 25  

In [8]:
df.drop(columns=['Volume','OHLCAvg','HLCAvg','HLAvg','BidVolume','AskVolume', '#ofTrades',
                 'ZeroLevel','ZeroLevel.1','TopLevel','BottomLevel','Line','Line.1','Line.2'], inplace=True) 

df.rename(columns={
    'Open': 'rawOpen',
    'High': 'rawHigh',
    'Low' : 'rawLow',
    'Last': 'rawLast',

    'Open.1': 'haOpen',
    'High.1': 'haHigh',
    'Low.1': 'haLow',
    'Last.1': 'haLast',
    
    'VEL.1': 'adpVEL',
    'TopBand': 'bolTopAdpVEL',
    'MiddleBand': 'bolMidAdpVEL',
    'BottomBand': 'bolBotAdpVEL',    
    'RSX.1': 'tickRSX',
    'Momentum': 'jmaD1',
    'Momentum.1': 'jmaD2',
    'TopBand.1': 'bolTopJmaD2',
    'MiddleBand.1': 'bolMidJmaD2',
    'BottomBand.1': 'bolBotJmaD2',
    'Momentum.2': 'tickJmaD1',
    'Momentum.3': 'tickJmaD2',
    'TopBand.2': 'bolTopTickJmaD2',
    'MiddleBand.2': 'bolMidTickJmaD2',
    'BottomBand.2': 'bolBotTickJmaD2',
    'JMA.1': 'tickJMA',
    'JMA.2': 'cfbJMA',
}, inplace=True)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11117814 entries, 0 to 11117813
Data columns (total 19 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   rawOpen    float64       
 2   rawHigh    float64       
 3   rawLow     float64       
 4   rawLast    float64       
 5   haOpen     float64       
 6   haHigh     float64       
 7   haLow      float64       
 8   haLast     float64       
 9   JMA        float64       
 10  VEL        float64       
 11  RSX        float64       
 12  jmaD1      float64       
 13  jmaD2      float64       
 14  tickJmaD1  float64       
 15  tickJmaD2  float64       
 16  tickJMA    float64       
 17  CFB        float64       
 18  cfbJMA     float64       
dtypes: datetime64[us](1), float64(18)
memory usage: 1.6 GB
None
            timestamp  rawOpen   rawHigh    rawLow   rawLast       haOpen    haHigh     haLow      haLast         JMA  VEL  RSX   jmaD1   jmaD2  tickJmaD1  tickJmaD2  tickJMA  CFB 

In [9]:
print(f'monotonic: {df["timestamp"].is_monotonic_increasing}')

df['date'] = df['timestamp'].dt.normalize()

filtered_days = [
   '2022-01-17', '2022-07-04', '2022-11-24', '2023-01-16',
   '2023-02-20', '2023-04-07', '2023-05-29', '2023-06-19',
   '2023-07-04', '2023-09-04', '2023-11-23', '2024-02-19',
   '2024-05-27', '2024-06-19', '2024-07-04', '2024-11-28',
   '2025-01-09', '2025-07-04', '2025-09-01', '2025-11-27',
   '2026-04-03', '2026-07-09',               '2025-11-28'
]

# before 1168
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print("Unique days before:", len(all_days))

df = df[
    ~df["timestamp"].dt.normalize().isin(pd.to_datetime(filtered_days))
]

# after -23 = 1145
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print(" Unique days after:", len(all_days))

#

print(df.info())
print(df.head())

monotonic: True
Unique days before: 1179
 Unique days after: 1156
<class 'pandas.DataFrame'>
Index: 11003371 entries, 0 to 11117813
Data columns (total 20 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   rawOpen    float64       
 2   rawHigh    float64       
 3   rawLow     float64       
 4   rawLast    float64       
 5   haOpen     float64       
 6   haHigh     float64       
 7   haLow      float64       
 8   haLast     float64       
 9   JMA        float64       
 10  VEL        float64       
 11  RSX        float64       
 12  jmaD1      float64       
 13  jmaD2      float64       
 14  tickJmaD1  float64       
 15  tickJmaD2  float64       
 16  tickJMA    float64       
 17  CFB        float64       
 18  cfbJMA     float64       
 19  date       datetime64[us]
dtypes: datetime64[us](2), float64(18)
memory usage: 1.7 GB
None
            timestamp  rawOpen   rawHigh    rawLow   rawLast       haOpen    haHigh     

In [10]:
df.to_parquet(outFile, index=False)
print(f'written to: {outFile}')

written to: data/mnq-6t-nq-3s-vol2-v2.pqt


In [11]:
print(df["timestamp"].dt.time.min())    
print(f'dates range: {df["date"].min()} .. {df["date"].max()}')    
x = 1/0

08:00:00
dates range: 2022-01-03 00:00:00 .. 2026-07-24 00:00:00


ZeroDivisionError: division by zero

In [ ]:
# find out why one is larger than the other by 1 day
outFile1= f'data/mnq-vol2-6tick-3sec-v0.pqt'
outFile2= f'data/mnq-vol2-6tick-3sec-v0-txt.pqt'

df1 = pd.read_parquet(outFile1)
df2 = pd.read_parquet(outFile2)

days1 = df1['date'].unique()
days2 = df2['date'].unique()

print(len(days1), len(days2))

missing = [x for x in days2 if x not in days1]
print(missing)

#csv = pd.read_parquet('csv.pqt')

print(df1.info())
print(df1.head())

print(df2.info())
print(df2.head())

df1.equals(df2)

In [ ]:
## WHY 3STREAM DONT MATCH BETWEEN OLD AND NEW
fsrc1 = f'data/mnq-tick-all-3sec.pqt'
fsrc2 = f'data/mnq-tick-full-3sec.pqt'
fraw = f'data/mnq-ohlc-raw-3sec.pqt'

src1 = pd.read_parquet(fsrc1)
src2 = pd.read_parquet(fsrc2)
raw  = pd.read_parquet(fraw)


In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.8f}')

print(src1.head(100))
display(src2.head())
display(raw.head())